# top 5 stop and search spots per force

uses the police stop and search csvs (the data.police.uk ones).
goes through all the monthly folders and works out the busiest spots for each force.

skipping greater-manchester, btp and northern ireland because of the redistribution boundary thing.

find_rate is just finds / searches (all categories mixed together, not per category)

In [ ]:
import os, glob
import pandas as pd
import numpy as np   

ROOT = r"C:\Users\20244270\OneDrive - TU Eindhoven\Desktop\CBL-midterm\2026-02"
TOP_N = 5

# forces we dont want
EXCLUDE = {"greater-manchester", "btp", "northern-ireland", "psni"}

cols = ["Latitude", "Longitude", "Outcome linked to object of search"]

## getting the files

In [ ]:
def get_force(p):
    fname = os.path.basename(p)
    # name is like 2024-05-essex-stop-and-search.csv
    # so cut off the date part (first 8 chars) and the ending
    fname = fname[8:]
    fname = fname.replace("-stop-and-search.csv", "")
    return fname

all_files = sorted(glob.glob(os.path.join(ROOT, "*", "*-stop-and-search.csv")))

files = []
for f in all_files:
    if get_force(f) not in EXCLUDE:
        files.append(f)

print(len(all_files), "files found,", len(files), "left after removing the excluded ones")
print(sorted(set(get_force(f) for f in files)))

In [ ]:
#read files
frames = []
for f in files:
    df = pd.read_csv(f, usecols=lambda c: c in cols)
    df["force"] = get_force(f)
    frames.append(df)

data = pd.concat(frames, ignore_index=True)
print("total rows:", len(data))

In [ ]:
# some rows have no lat/long, drop 
data = data.dropna(subset=["Latitude", "Longitude"])

# make a True/False column for whether they actually found something
# anything thats not 'true' counts as not a find
data["linked"] = data["Outcome linked to object of search"].astype(str).str.lower() == "true"

In [ ]:
#count searches at each spot
counts = data.groupby(["force", "Latitude", "Longitude"]).agg(
    searches=("linked", "size"),
    linked_finds=("linked", "sum"),
).reset_index()

counts["find_rate"] = counts["linked_finds"] / counts["searches"]
counts.head()

## top 5 for each force

In [ ]:
# sort by force, then most searches first
counts = counts.sort_values(["force", "searches"], ascending=[True, False])

top5 = counts.groupby("force").head(TOP_N).reset_index(drop=True)

# rank them 1 to 5
top5["rank"] = top5.groupby("force").cumcount() + 1

top5 = top5[["force", "rank", "Latitude", "Longitude", "searches", "linked_finds", "find_rate"]]
print(top5["force"].nunique(), "forces,", len(top5), "rows")
top5

## save

In [ ]:
top5.to_csv("top5_locations_by_force.csv", index=False)
print("saved")

# also save just the locations (for the map later)
top5[["force", "Latitude", "Longitude"]].to_csv("top5_only_loc.csv", index=False)